In [13]:
import jax
import jax.numpy as jnp
import netket as nk
import netket.experimental as nkx
import numpy as np
from pyscf import gto, scf, fci
import flax.linen as nn
from textwrap import dedent

#==============================================================================
# 1. H2 分子 & FCI 基准
#==============================================================================
K = 2
bond_length = 1.4
geometry = [('H', (0.,0.,0.)), ('H', (bond_length,0.,0.))]
mol = gto.M(atom=geometry, basis='STO-3G', verbose=0)
mf = scf.RHF(mol).run(verbose=0)

# FCI 精确解
cisolver = fci.FCI(mf)
cisolver.nroots = 4
E_fcis, _ = cisolver.kernel()
print("="*60)
print("FCI 能量")
print("="*60)
for i, e in enumerate(E_fcis):
    print(f"E{i} = {e:.8f} Ha")

# 原始系统哈密顿
ha_original = nkx.operator.from_pyscf_molecule(mol)
H_mat = ha_original.to_dense()  # (4,4) 矩阵，用于计算局域能量

#==============================================================================
# 2. 希尔伯特空间：扩展系统 hi^K
#==============================================================================
hi = nk.hilbert.SpinOrbitalFermions(
    n_orbitals=2,
    s=1/2,
    n_fermions_per_spin=(1,1)
)
hi_K = hi ** K  # 扩展希尔伯特空间（NES 核心）

#==============================================================================
# 3. 采样器：扩展空间费米子采样
#==============================================================================
g = nk.graph.Graph(edges=[(0,1), (2,3)])
single_rule = nk.sampler.rules.FermionHopRule(hilbert=hi, graph=g)
tensor_rule = nk.sampler.rules.TensorRule(hilbert=hi_K, rules=[single_rule]*K)

sampler = nk.sampler.MetropolisSampler(
    hi_K,
    rule=tensor_rule,
    n_chains=16,
    sweep_size=64
)

#==============================================================================
# ✅ NES-VMC 核心：行列式波函数 + RBM 替换神经网络
#==============================================================================
class NES_RBM(nn.Module):
    K: int
    n_spin_orb: int = 4
    alpha: int = 1

    def setup(self):
        # K 个独立 RBM → 替换你的 SingleStateAnsatz
        self.rbms = [
            nk.models.RBM(alpha=self.alpha, param_dtype=complex)
            for _ in range(self.K)
        ]

    @nn.compact
    def __call__(self, x):
        log_psi, _ = self.compute_logpsi_M(x)
        return log_psi

    def compute_logpsi_M(self, x):
        x_reshaped = x.reshape(-1, self.K, self.n_spin_orb)

        def make_M(xs):
            rows = []
            for i in range(self.K):
                row = [self.rbms[j](xs[i][None, ...])[0] for j in range(self.K)]
                rows.append(jnp.array(row))
            return jnp.stack(rows)

        M = jax.vmap(make_M)(x_reshaped)
        psi = jnp.linalg.det(M)
        log_psi = jnp.log(psi)
        return log_psi, M

#==============================================================================
# 4. 模型 & MCState
#==============================================================================
model = NES_RBM(K=K)
mc_state = nk.vqs.MCState(
    sampler=sampler,
    model=model,
    n_samples=512,
    n_discard_per_chain=10
)

#==============================================================================
# ✅ 关键修复：自定义损失函数，不使用 ha 直接驱动！
#==============================================================================
opt = nk.optimizer.Sgd(learning_rate=0.05)
sr = nk.optimizer.SR(diag_shift=0.01, holomorphic=True)

# 自定义损失：扩展系统能量 = Tr( E_L_avg )
def total_energy():
    samples = mc_state.sample()
    _, M = model.compute_logpsi_M(samples)
    E_L = local_energy_matrix(H_mat, M)
    E_L_avg = jnp.mean(E_L, axis=0)
    return jnp.trace(E_L_avg).real

# 局域能量矩阵
@jax.jit
def local_energy_matrix(H_mat, M):
    def apply_H(m): return H_mat @ m
    HM = jax.vmap(apply_H)(M)
    psi = jnp.linalg.det(M)
    return HM / psi.reshape(-1,1,1)

#==============================================================================
# 手动训练循环（彻底解决希尔伯特空间不匹配）
#==============================================================================
print("\n" + "="*50)
print("NES-VMC 训练开始 (RBM 版)")
print("="*50)

for step in range(300):
    # 一步梯度下降
    mc_state.advance(total_energy, n_steps=1, optimizer=opt, preconditioner=sr)

    if step % 50 == 0:
        samples = mc_state.sample()
        _, M = model.compute_logpsi_M(samples)
        E_L = local_energy_matrix(H_mat, M)
        E_L_avg = jnp.mean(E_L, axis=0)
        eig_vals = jnp.sort(jnp.linalg.eigvalsh(E_L_avg).real)

        print(f"\nStep {step}")
        for i, e in enumerate(eig_vals):
            print(f"E{i} = {e:.8f} Ha")

#==============================================================================
# 最终结果
#==============================================================================
print("\n" + "="*50)
print("NES-VMC 最终结果")
print("="*50)

samples = mc_state.sample()
_, M = model.compute_logpsi_M(samples)
E_L_avg = jnp.mean(local_energy_matrix(H_mat, M), axis=0)
eig_vals = jnp.sort(jnp.linalg.eigvalsh(E_L_avg).real)

for i, e in enumerate(eig_vals):
    print(f"E{i} = {e:.8f} Ha")

print("\nFCI 对比:")
for i in range(K):
    print(f"E{i} = {E_fcis[i]:.8f} Ha")

FCI 能量
E0 = -1.01546825 Ha
E1 = -0.87542794 Ha
E2 = -0.42938376 Ha
E3 = -0.26922131 Ha

NES-VMC 训练开始 (RBM 版)


AttributeError: 'MCState' object has no attribute 'advance'

In [14]:
import jax
import jax.numpy as jnp
import netket as nk
import netket.experimental as nkx
import numpy as np
from pyscf import gto, scf, fci
import flax.linen as nn

#==============================================================================
# 1. H2 分子 & FCI 基准
#==============================================================================
K = 2
bond_length = 1.4
geometry = [('H', (0.,0.,0.)), ('H', (bond_length,0.,0.))]
mol = gto.M(atom=geometry, basis='STO-3G', verbose=0)
mf = scf.RHF(mol).run(verbose=0)

cisolver = fci.FCI(mf)
cisolver.nroots = 4
E_fcis, _ = cisolver.kernel()
print("="*60)
print("FCI 精确能量")
print("="*60)
for i, e in enumerate(E_fcis):
    print(f"E{i} = {e:.8f} Ha")

ha_original = nkx.operator.from_pyscf_molecule(mol)
H_mat = ha_original.to_dense()  # 原始系统哈密顿矩阵 (4,4)

#==============================================================================
# 2. 扩展希尔伯特空间 hi^K
#==============================================================================
hi = nk.hilbert.SpinOrbitalFermions(
    n_orbitals=2,
    s=1/2,
    n_fermions_per_spin=(1,1)
)
hi_K = hi ** K

#==============================================================================
# 3. 扩展空间采样器
#==============================================================================
g = nk.graph.Graph(edges=[(0,1), (2,3)])
single_rule = nk.sampler.rules.FermionHopRule(hilbert=hi, graph=g)
tensor_rule = nk.sampler.rules.TensorRule(hilbert=hi_K, rules=[single_rule]*K)

sampler = nk.sampler.MetropolisSampler(
    hi_K,
    rule=tensor_rule,
    n_chains=16,
    sweep_size=64
)

#==============================================================================
# ✅ NES-VMC：RBM 行列式波函数
#==============================================================================
class NES_RBM(nn.Module):
    K: int
    n_spin_orb: int = 4
    alpha: int = 1

    def setup(self):
        self.rbms = [nk.models.RBM(alpha=self.alpha, param_dtype=complex) for _ in range(self.K)]

    @nn.compact
    def __call__(self, x):
        x_r = x.reshape(-1, self.K, self.n_spin_orb)

        def make_M(xs):
            rows = []
            for i in range(self.K):
                row = [self.rbms[j](xs[i][None, :])[0] for j in range(self.K)]
                rows.append(jnp.array(row))
            return jnp.stack(rows)

        M = jax.vmap(make_M)(x_r)
        psi = jnp.linalg.det(M)
        return jnp.log(psi)

#==============================================================================
# 4. 模型 & MCState
#==============================================================================
model = NES_RBM(K=K)
mc_state = nk.vqs.MCState(
    sampler=sampler,
    model=model,
    n_samples=512,
    n_discard_per_chain=10
)

#==============================================================================
# ✅ 核心：自定义能量算子（解决希尔伯特空间不匹配）
#==============================================================================
def energy_fn(x):
    x_r = x.reshape(-1, K, 4)
    def make_M(xs):
        rows = [[mc_state.model.rbms[j](xs[i][None, :])[0] for j in range(K)] for i in range(K)]
        return jnp.stack(rows)
    M = jax.vmap(make_M)(x_r)
    HM = jax.vmap(lambda m: H_mat @ m)(M)
    psi = jnp.linalg.det(M)
    Eloc_mat = HM / psi.reshape(-1,1,1)
    return jnp.trace(Eloc_mat).real

# 包装为 NetKet 算子
custom_energy = nk.operator.GenericOperator(hi_K, energy_fn)

#==============================================================================
# 5. 标准 VMC 驱动（无任何报错）
#==============================================================================
opt = nk.optimizer.Sgd(learning_rate=0.05)
sr = nk.optimizer.SR(diag_shift=0.01, holomorphic=True)

vmc = nk.driver.VMC(
    custom_energy,
    opt,
    variational_state=mc_state,
    preconditioner=sr
)

#==============================================================================
# 6. 提取 M 矩阵 & 对角化激发态
#==============================================================================
def get_M(samples):
    x_r = samples.reshape(-1, K, 4)
    def make_M(xs):
        rows = [[mc_state.model.rbms[j](xs[i][None, :])[0] for j in range(K)] for i in range(K)]
        return jnp.stack(rows)
    return jax.vmap(make_M)(x_r)

@jax.jit
def local_energy_matrix(M):
    HM = jax.vmap(lambda m: H_mat @ m)(M)
    psi = jnp.linalg.det(M)
    return HM / psi.reshape(-1,1,1)

#==============================================================================
# 训练
#==============================================================================
print("\n" + "="*50)
print("NES-VMC 训练开始")
print("="*50)

for step in range(300):
    vmc.advance(1)

    if step % 50 == 0:
        samples = mc_state.sample()
        M = get_M(samples)
        E_L = local_energy_matrix(M)
        E_L_avg = jnp.mean(E_L, axis=0)
        eig_vals = jnp.sort(jnp.linalg.eigvalsh(E_L_avg).real)

        print(f"\nStep {step}")
        for i, e in enumerate(eig_vals):
            print(f"E{i} = {e:.8f} Ha")

#==============================================================================
# 最终结果
#==============================================================================
print("\n" + "="*50)
print("NES-VMC 最终结果")
print("="*50)

samples = mc_state.sample()
M = get_M(samples)
E_L_avg = jnp.mean(local_energy_matrix(M), axis=0)
eig_vals = jnp.sort(jnp.linalg.eigvalsh(E_L_avg).real)

for i, e in enumerate(eig_vals):
    print(f"E{i} = {e:.8f} Ha")

print("\nFCI 对比:")
for i in range(K):
    print(f"E{i} = {E_fcis[i]:.8f} Ha")

FCI 精确能量
E0 = -1.01546825 Ha
E1 = -0.87542794 Ha
E2 = -0.42938376 Ha
E3 = -0.26922131 Ha


AttributeError: module 'netket.operator' has no attribute 'GenericOperator'

In [15]:
import jax
import jax.numpy as jnp
import netket as nk
import netket.experimental as nkx
import numpy as np
from pyscf import gto, scf, fci
from flax import nnx
import optax
from tqdm import tqdm

#==============================================================================
# 1. 全局参数 & H₂ 分子系统定义
#==============================================================================
K = 2  # 计算前 K 个态（基态 + K-1 个激发态）
bond_length = 1.4
geometry = [('H', (0., 0., 0.)), ('H', (bond_length, 0., 0.))]
mol = gto.M(atom=geometry, basis='STO-3G', verbose=0)
mf = scf.RHF(mol).run(verbose=0)

# FCI 精确基准（用于对比）
cisolver = fci.FCI(mol)
cisolver.nroots = 4
E_fcis, fcivec = cisolver.kernel()
print("="*60)
print("H₂ FCI 基准能量")
print("="*60)
for i, e in enumerate(E_fcis):
    exc = (e - E_fcis[0]) * 27.2114
    print(f"E{i} = {e:.8f} Ha  |  激发能: {exc:.4f} eV")

# 转换为 NetKet 哈密顿量
ha = nkx.operator.from_pyscf_molecule(mol)

# 原始 Hilbert 空间（自旋轨道费米子）
hi = nk.hilbert.SpinOrbitalFermions(
    n_orbitals=2,
    s=1/2,
    n_fermions_per_spin=(1, 1),
)

# 扩展 Hilbert 空间：hi^K（直接用 ** 运算符，无需重复 TensorHilbert）
hi_ensemble = hi ** K

# 采样器设计：TensorRule 组合 K 个 FermionHopRule
edges = [(0, 1), (2, 3)]  # 原始 Hilbert 空间的跃迁图
g = nk.graph.Graph(edges=edges)
single_rule = nk.sampler.rules.FermionHopRule(hilbert=hi, graph=g)
tensor_rule = nk.sampler.rules.TensorRule(hilbert=hi_ensemble, rules=[single_rule]*K)
sampler = nk.sampler.MetropolisSampler(hi_ensemble, rule=tensor_rule, n_chains=16, sweep_size=32)


#==============================================================================
# 2. 神经网络 Ansatz 设计
#==============================================================================
class SingleStateAnsatz(nnx.Module):
    """单个态的波函数 ansatz ψ_i(x)"""
    n_spin_orbitals: int
    hidden_dim: int = 16

    @nnx.compact  # 自动管理参数初始化
    def __call__(self, x):
        x = x.astype(complex)
        x = nnx.Linear(self.n_spin_orbitals, self.hidden_dim, param_dtype=complex)(x)
        x = nnx.tanh(x)
        x = nnx.Linear(self.hidden_dim, self.hidden_dim, param_dtype=complex)(x)
        x = nnx.tanh(x)
        x = nnx.Linear(self.hidden_dim, 1, param_dtype=complex)(x)
        return jnp.squeeze(x)  # 输出复数标量 ψ_i(x)


class NESTotalAnsatz(nnx.Module):
    """NES-VMC 扩展系统的总波函数 ansatz Ψ(x¹,...,xᵏ) = det(M)"""
    n_spin_orbitals: int
    n_states: int = K
    hidden_dim: int = 16

    def setup(self):
        # 初始化 K 个独立的 SingleStateAnsatz（不同随机种子）
        keys = jax.random.split(jax.random.key(42), self.n_states)
        self.single_ansatz_list = [
            SingleStateAnsatz(
                n_spin_orbitals=self.n_spin_orbitals,
                hidden_dim=self.hidden_dim
            )
            for _ in range(self.n_states)
        ]

    def __call__(self, x):
        """
        输入: x (n_spin_orbitals*K,) → 扩展系统的单个样本
        输出: log_psi (复数标量) → log(det(M))，供采样器使用
              M (K, K) → 矩阵 M，供局域能量计算
        """
        # 将 x 重塑为 (K, n_spin_orbitals)，对应 x¹,...,xᵏ
        x_batch = x.reshape(self.n_states, self.n_spin_orbitals)

        # 构建矩阵 M: M[i][j] = ψ_j(xⁱ)
        M = []
        for i in range(self.n_states):
            row = [self.single_ansatz_list[j](x_batch[i]) for j in range(self.n_states)]
            M.append(jnp.stack(row))
        M = jnp.stack(M)  # (K, K) 复数矩阵

        # 数值稳定的对数行列式计算：log(det(M)) = log|det| + log(sign)
        sign, log_abs_det = jnp.linalg.slogdet(M)
        log_psi = log_abs_det + jnp.log(sign.astype(complex))

        return log_psi, M


#==============================================================================
# 3. 局域能量与损失函数计算
#==============================================================================
def Ham_psi(ha: nk.operator.DiscreteOperator, single_ansatz: SingleStateAnsatz, x: jnp.array):
    """计算 Ĥψ_i(x) = sum_{x'} H(x, x')ψ_i(x')"""
    x_primes, mels = ha.get_conn_padded(x)  # 连接构型与矩阵元
    psi_vals = jax.vmap(single_ansatz)(x_primes)  # 批量计算 ψ_i(x')
    return jnp.sum(mels * psi_vals)


def Ham_Psi(ha: nk.operator.DiscreteOperator, total_ansatz: NESTotalAnsatz, x_batch):
    """构建矩阵 ĤΨ: (ĤΨ)[i][j] = Ĥψ_j(xⁱ)"""
    K_state = total_ansatz.n_states
    H_mat = []
    for i in range(K_state):
        row = [Ham_psi(ha, total_ansatz.single_ansatz_list[j], x_batch[i]) for j in range(K_state)]
        H_mat.append(jnp.stack(row))
    return jnp.stack(H_mat)


def compute_local_energy(ha: nk.operator.DiscreteOperator, total_ansatz: NESTotalAnsatz, x):
    """计算单个扩展样本的局域能量矩阵 E_L = M⁻¹ĤΨ 及其迹"""
    x_batch = x.reshape(total_ansatz.n_states, total_ansatz.n_spin_orbitals)
    log_psi, M = total_ansatz(x)

    # 数值稳定性：给 M 加小对角矩阵避免奇异
    eps = 1e-6
    M_reg = M + eps * jnp.eye(M.shape[0], dtype=M.dtype)

    # 计算 ĤΨ 与局域能量矩阵
    Hp = Ham_Psi(ha, total_ansatz, x_batch)
    el_mat = jnp.linalg.solve(M_reg, Hp)
    tr_el = jnp.trace(el_mat)

    return tr_el.real, el_mat  # 迹取实数，el_mat 保留复数


# 向量化批量计算局域能量
compute_local_energy_batch = jax.vmap(
    compute_local_energy,
    in_axes=(None, None, 0),  # 仅对样本 x 向量化
    out_axes=(0, 0)
)


def compute_average_local_energy(ha: nk.operator.DiscreteOperator, total_ansatz: NESTotalAnsatz, samples):
    """计算批量样本的平均局域能量矩阵与迹"""
    tr_els, el_mats = compute_local_energy_batch(ha, total_ansatz, samples)
    tr_avg = jnp.mean(tr_els)
    el_mat_avg = jnp.mean(el_mats, axis=0)
    return tr_avg, el_mat_avg


# 采样器兼容的模型前向函数（仅返回 log_psi）
def model_forward(parameters, x):
    graphdef, variables = parameters
    model = nnx.merge(graphdef, variables)
    log_psi, _ = model(x)
    return log_psi


# 损失函数：平均局域能量的迹（扩展系统的基态能量）
def loss_fn(parameters, ha, samples):
    graphdef, variables = parameters
    model = nnx.merge(graphdef, variables)
    tr_avg, _ = compute_average_local_energy(ha, model, samples)
    return tr_avg


# 损失与梯度计算函数
value_and_grad_fn = jax.value_and_grad(loss_fn)


#==============================================================================
# 4. 训练循环
#==============================================================================
# 初始化模型与参数
n_spin_orbitals = hi.size  # 4
total_ansatz = NESTotalAnsatz(n_spin_orbitals=n_spin_orbitals, n_states=K, hidden_dim=16)
graphdef, variables = nnx.split(total_ansatz)  # 拆分模型结构与参数
parameters = (graphdef, variables)

# 优化器与采样器状态初始化
optimizer = optax.adam(learning_rate=1e-3)
opt_state = optimizer.init(variables)
sampler_state = sampler.init_state(model_forward, parameters, seed=42)

# 训练记录
loss_record = []
el_mat_record = []
n_steps = 200
chain_length = 200

print("\n" + "="*60)
print("开始 NES-VMC 训练")
print("="*60)

for step in tqdm(range(n_steps)):
    # 1. 重置采样器并生成新样本
    sampler_state = sampler.reset(model_forward, parameters, state=sampler_state)
    samples, sampler_state = sampler.sample(
        model_forward, parameters, state=sampler_state, chain_length=chain_length
    )
    samples_flat = samples.reshape(-1, hi_ensemble.size)  # 展平为 (n_samples, n_spin_orbitals*K)

    # 2. 计算损失与梯度
    loss, grads = value_and_grad_fn(parameters, ha, samples_flat)
    loss_record.append(loss)

    # 3. 更新模型参数（仅更新 variables，保留 graphdef）
    grad_graphdef, grad_variables = grads
    updates, opt_state = optimizer.update(grad_variables, opt_state, variables)
    variables = optax.apply_updates(variables, updates)
    parameters = (graphdef, variables)

    # 4. 定期打印结果与记录局域能量矩阵
    if step % 20 == 0:
        current_model = nnx.merge(graphdef, variables)
        _, current_el_mat = compute_average_local_energy(ha, current_model, samples_flat)
        el_mat_record.append(current_el_mat)
        current_eigen = jnp.linalg.eigvalsh(current_el_mat).real

        print(f"\nStep {step:4d} | Loss (Tr(E_L)): {loss:.8f} Ha")
        print(f"  当前本征能量: {current_eigen}")
        print(f"  FCI 基准能量: {E_fcis[:K]}")


#==============================================================================
# 5. 最终结果计算
#==============================================================================
# 生成更多样本以获得更精确的平均局域能量矩阵
final_model = nnx.merge(graphdef, variables)
sampler_state = sampler.reset(model_forward, parameters, state=sampler_state)
final_samples, _ = sampler.sample(
    model_forward, parameters, state=sampler_state, chain_length=1000
)
final_samples_flat = final_samples.reshape(-1, hi_ensemble.size)

# 计算最终平均局域能量矩阵并对角化
_, final_el_mat = compute_average_local_energy(ha, final_model, final_samples_flat)
final_eigen_energies = jnp.linalg.eigvalsh(final_el_mat).real

print("\n" + "="*60)
print("最终 NES-VMC 结果")
print("="*60)
print(f"NES-VMC 前 {K} 个能量: {final_eigen_energies}")
print(f"FCI 基准前 {K} 个能量: {E_fcis[:K]}")
print("="*60)

TypeError: FCIBase.kernel() missing 4 required positional arguments: 'h1e', 'eri', 'norb', and 'nelec'